# 第2回 演習: LoRA で回答形式を学習し、学習前後を比較する

架空の講座 FAQ を題材に、小さな指示チューニング済みモデルを LoRA で短時間学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。評価は **形式遵守率**（`answer` と `needs_review` の JSON を守れたか）と **内容正解率** に分けます。

**この演習で確かめること**

1. プロンプトで形式を指示しても、小さなモデルは守れないことがある（セクション 3、学習前）。
2. LoRA で数分学習すると、形式はほぼ守れるようになる（セクション 5、学習後）。
3. 形式が整っても、FAQ に無いことを聞かれたときの判断（確認を求めるか、断定してしまうか）までは保証されない（セクション 5、6）。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、データ生成、モデル読み込み、学習前評価 | 約 5 分 |
| 4〜7 | 学習、学習後評価、知識質問、比較表と提出 | 約 10 分（学習 4〜5 分を含む） |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。データはこのノートブックが生成する架空データで、実在の締切や URL は含みません。

**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.1（2026-10-18）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 架空 FAQ と教師データの生成

**やること**: 5 項目の架空 FAQ から、学習用（train）、検証用（valid）、評価用（test）のデータを作ります。

モデルへの入力は「FAQ ＋ 質問」、望ましい出力は `{"answer": <文字列>, "needs_review": <真偽値>}` の JSON 1 行です。

- FAQ に根拠がある質問には、FAQ の文で答え、`needs_review: false` にする。
- FAQ に無い質問には、「この FAQ には記載がありません。運営に確認が必要です。」と答え、`needs_review: true` にする。

**データの分け方**

- 同じ FAQ 項目を聞く質問の言い換えを、**テンプレート単位**で train / valid / test に振り分けます。学習で見た文がそのまま test に出ないようにするためです。
- test は 21 問（根拠あり 15、根拠なし 6）です。件数を確保するため、test にだけ使う質問を別に用意しています。
- 正解（`needs_review` の値と、答えに含まれるべき語句）は `gold` に分けて持ち、モデルには渡しません。
- train は 14 件（根拠あり 10、根拠なし 4）、valid は 7 件です。FAQ が 5 項目しかないので学習データは少なく、同じ例を何回も学習することになります。何回学習するかは、データを複製して増やすのではなく、セクション 4 のエポック数（`num_train_epochs`）で決めます。データのファイルには実例だけを入れ、学習量は学習の設定で持つのが基本です。

**見るところ**: `train=14 valid=7 test=21 gold=21` と、学習例の 1 件（質問と、望ましい出力）が表示されます。質問が train / valid / test のどれかで重複していると、ここでエラーになります。

In [ ]:
FAQ = {
    "deadline": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
    "destination": "提出先は学習ポータルの「課題」欄です。",
    "format": "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
    "office_hours": "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
    "late": "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# 根拠あり: (FAQ キー, 質問テンプレート群, 正解の answer に含まれるべき語)
GROUNDED = {
    "deadline": (["課題の提出期限はいつですか", "いつまでに出せばいいですか", "締切を教えてください", "提出の期限は？"], ["9 月 30 日", "23:59"]),
    "destination": (["提出先はどこですか", "どこにアップロードしますか", "課題はどこから出しますか", "提出場所は？"], ["学習ポータル", "課題"]),
    "format": (["提出形式を教えてください", "ファイル名の決まりはありますか", "何のファイル形式で出しますか", "PDF 以外でも出せますか"], ["PDF"]),
    "office_hours": (["質問はいつできますか", "オフィスアワーはいつですか", "先生に質問する時間はありますか", "相談できる時間帯は？"], ["火曜", "17:00"]),
    "late": (["遅れて出したらどうなりますか", "期限を過ぎた場合は受け付けますか", "遅延提出のルールは？", "締切後の扱いを教えてください"], ["48 時間", "減点"]),
}
# 根拠なし: FAQ に無いので needs_review=true が正解
UNGROUNDED = [
    "再提出はできますか", "欠席した回の扱いを教えてください", "他の受講者の点数は見られますか",
    "成績はいつ出ますか", "提出期限は延長できますか", "グループでの提出は可能ですか",
    "課題の配点はどうなっていますか", "講義の録画は公開されますか",
]

# test 専用の追加質問（train / valid には入れない）
TEST_EXTRA_GROUNDED = {
    "deadline": ["提出期限の日付と時刻を教えてください", "課題の締切は何月何日の何時ですか"],
    "destination": ["課題の提出はどこで行いますか", "提出物を出す先を教えてください"],
    "format": ["提出するファイルの形式は何ですか", "提出ファイルは何形式で、いくつ出しますか"],
    "office_hours": ["オフィスアワーの曜日と時間を教えてください", "質問を受け付けてもらえるのは何曜日の何時からですか"],
    "late": ["提出が締切に間に合わなかったらどうなりますか", "期限後に提出した課題は受理されますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "課題の文字数に制限はありますか", "提出した課題へのフィードバックはもらえますか",
    "参考文献の書き方に指定はありますか", "オフィスアワーは事前予約が必要ですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA 学習（T4 で 4〜5 分）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは 10 ステップごとに `/content/qlora_lab/ckpt/` に保存されています。ファイルが残っていれば、セクション 0〜3 を実行し直してから、上の学習セルの `trainer.train()` を `trainer.train(resume_from_checkpoint=True)` に書き換えて実行すると、最後のチェックポイントから 40 ステップ（20 エポック）まで続きを学習します。ランタイムごと消えてファイルが無い場合は、学習をやり直すか、下の方法に切り替えます。

**学習できなかった（GPU が確保できない、時間内に終わらない）**: 講師が同じノートブックを T4 で実行した結果（学習後の出力とスコア）を使って、比較と考察を進めます。セクション 5 以降のセルは学習後のモデル `tuned` が無いと動かないので、講師の結果を見ながら説明を読んでください。提出するときは、セクション 7 の `executed_training_myself` を `False` にし、本人が実行できた範囲を明記します。

## 5. 学習後評価（学習前と同じ test・同じ生成設定）

**やること**: 学習後のモデル `tuned` で、学習前とまったく同じ 21 問・同じ生成設定で評価し、前後を並べます。変えたのは adapter の有無だけです。

**見るところ**

- 比較表: `format_rate`（形式遵守率）と `content_rate`（内容正解率）が学習前からどう変わったか。
- `退行`: 学習前は形式も内容も通っていたのに、学習後に落ちた質問です。空のリストが期待値です。
- 質問ごとの表: 残った失敗を読み、どの型かを見分けてください。
  - 根拠なしの質問（正解の `needs_review` が `True`）に、FAQ の別項目を当てはめて断定した。
  - 根拠ありの質問（正解が `False`）に、「記載がありません」と確認要求で逃げた。

学習後の回答は、教師データの文（FAQ の原文）をほぼそのまま返す形になります。形式と文面は揃いますが、「FAQ に根拠があるか」の判断まで学習できたとは限りません。

seed を固定しても、GPU 上の学習は完全には再現しません。形式遵守率と内容正解率の割合はほぼ同じでも、どの質問が残るかは実行ごとに変わることがあります。隣の人と「残った質問」が違っても異常ではないので、失敗の型で比べてください。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 知識を問う質問（形式を学んでも知識は保証されない）

**やること**: 形式の実験とは別に、FAQ にも学習データにも無い事実を 3 問尋ねます（対照として FAQ にある質問を 1 問加えます）。望ましい応答は「確認が必要（`needs_review: true`）」です。

応答を 3 つに分類します。

| 分類 | 意味 |
| --- | --- |
| `format_error` | JSON の形式を守れていない（学習前に多い） |
| `needs_review` | 「記載がありません。確認が必要です」と返した（望ましい応答） |
| `asserted` | `needs_review: false` で言い切った。FAQ に無い質問に対してなら、根拠のない断定（創作）です |

**見るところ**: 学習前は形式違反になりやすく、学習後は「確認が必要」と「断定」の両方が出ることがあります。出力の原文も表に出すので、断定した場合に何を根拠にしたように見えるか（FAQ の別項目を流用しているか）を読んでください。「Fine-Tuning では知識は絶対に直らない」と先に結論にせず、今回のデータでの結果を報告します。

学習前の出力は、adapter を一時的に無効にして取っています（`tuned.disable_adapter()`）。

In [ ]:
KNOWLEDGE_QUESTIONS = [
    "この講座の最終課題の締切はいつですか",       # FAQ に無い（架空）
    "担当教員の名前は何ですか",                     # FAQ に無い
    "次回の講義のテーマは何ですか",                 # FAQ に無い
    "課題の提出期限はいつですか",                   # FAQ にある（対照）
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "FAQ にある": q in gold, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                 "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. 比較表と提出

**やること**: 設定、ライブラリの版、学習前後の要約、退行した質問を 1 つの `result.json` にまとめます。

**提出するもの**

- `result.json`、`eval_before.json`、`eval_after.json`（`/content/qlora_lab` にあります。左のファイル一覧で右クリック → ダウンロード。`USE_DRIVE = True` なら Drive にもコピーされます）
- セクション 5 の比較表と、残った失敗の説明（どの型の失敗か）
- セクション 6 の結果（確認要求に変わった応答、断定が残った応答）
- 本人が学習を実行した範囲。講師の結果を使った場合は `executed_training_myself` を `False` にしてください。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

# 発展課題：用語集の学習と検索との比較
前半は講座の演習、後半は独立した発展課題です。全セルを上から実行します。
学習や評価はこのノートブックの実行時に初めて行われます。既存の結果は含みません。
後半の学習は120例×3エポック=45更新を予定します。演習の14例×20エポック=40更新とはデータ量が違うため、時間も同じとは限りません。
T4 GPUを使用してください。講座のPINSとモデルrevisionは変更しません。


In [ ]:

# 演習の結果を別の場所に保存してからモデルを解放する
import shutil, gc
EXERCISE_DIR = Path('/content/lesson02_exercise'); EXERCISE_DIR.mkdir(exist_ok=True)
for name in ['result.json','eval_before.json','eval_after.json','test.jsonl','gold.json']:
    shutil.copy(WORK/name, EXERCISE_DIR/name)
EXERCISE_RESULT = json.loads((EXERCISE_DIR/'result.json').read_text())
del trainer, tuned, model
if 'model_4bit' in globals(): del model_4bit
gc.collect(); torch.cuda.empty_cache()
assert torch.cuda.is_available(), 'T4 GPUランタイムで実行してください。'


In [ ]:
PAYLOAD = json.loads('{"terms": [{"id": "ai-engineering", "term": "AIエンジニアリング", "aliases": ["AI Engineering"], "lecture": 1, "category": "全体像", "definition": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。", "related": ["llm", "evaluation-loop"]}, {"id": "evaluation-loop", "term": "評価と改善のサイクル", "aliases": ["改善サイクル"], "lecture": 1, "category": "全体像", "definition": "作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。", "related": ["ai-engineering", "hallucination"]}, {"id": "llm", "term": "LLM（大規模言語モデル）", "aliases": ["大規模言語モデル", "Large Language Model"], "lecture": 1, "category": "LLM基礎", "definition": "大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。", "related": ["token", "prompt", "pretraining"]}, {"id": "token", "term": "トークン", "aliases": ["token"], "lecture": 1, "category": "LLM基礎", "definition": "LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。", "related": ["context-window", "rate-limit"]}, {"id": "context-window", "term": "コンテキストウィンドウ", "aliases": ["コンテキスト長", "context window"], "lecture": 1, "category": "LLM基礎", "definition": "LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。", "related": ["token", "agent-loop"]}, {"id": "prompt", "term": "プロンプト", "aliases": ["prompt"], "lecture": 1, "category": "LLM基礎", "definition": "LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。", "related": ["system-prompt", "llm"]}, {"id": "system-prompt", "term": "システムプロンプト", "aliases": ["指示文", "instructions", "system instruction"], "lecture": 1, "category": "LLM基礎", "definition": "利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。", "related": ["prompt", "guardrail"]}, {"id": "temperature", "term": "temperature", "aliases": ["温度パラメータ"], "lecture": 1, "category": "LLM基礎", "definition": "次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。", "related": ["llm", "evaluation-loop"]}, {"id": "hallucination", "term": "ハルシネーション", "aliases": ["幻覚", "hallucination"], "lecture": 1, "category": "LLM基礎", "definition": "LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。", "related": ["tool", "evaluation-loop"]}, {"id": "api", "term": "API", "aliases": ["Application Programming Interface", "Web API"], "lecture": 1, "category": "開発基礎", "definition": "プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。", "related": ["api-key", "rate-limit"]}, {"id": "api-key", "term": "APIキー", "aliases": ["API key"], "lecture": 1, "category": "開発基礎", "definition": "APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。", "related": ["codespaces-secret", "environment-variable"]}, {"id": "environment-variable", "term": "環境変数", "aliases": ["environment variable", "env"], "lecture": 1, "category": "開発基礎", "definition": "プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。", "related": ["api-key", "codespaces-secret"]}, {"id": "rate-limit", "term": "レート制限", "aliases": ["rate limit", "429エラー"], "lecture": 1, "category": "開発基礎", "definition": "一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。", "related": ["api", "agent-loop", "default-model"]}, {"id": "codespaces", "term": "GitHub Codespaces", "aliases": ["Codespaces", "コードスペース"], "lecture": 1, "category": "開発環境", "definition": "GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。", "related": ["devcontainer", "codespaces-secret"]}, {"id": "devcontainer", "term": "devcontainer", "aliases": ["Dev Container", "devcontainer.json"], "lecture": 1, "category": "開発環境", "definition": "開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。", "related": ["codespaces"]}, {"id": "codespaces-secret", "term": "Codespaces secret", "aliases": ["Codespacesのシークレット"], "lecture": 1, "category": "開発環境", "definition": "GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。", "related": ["api-key", "environment-variable", "codespaces"]}, {"id": "fork", "term": "fork", "aliases": ["フォーク"], "lecture": 1, "category": "Git/GitHub", "definition": "他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。", "related": ["branch", "pull-request"]}, {"id": "branch", "term": "ブランチ", "aliases": ["branch"], "lecture": 1, "category": "Git/GitHub", "definition": "作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。", "related": ["pull-request", "fork"]}, {"id": "pull-request", "term": "プルリクエスト（PR）", "aliases": ["PR", "Pull Request"], "lecture": 1, "category": "Git/GitHub", "definition": "あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。", "related": ["branch", "answer-md"]}, {"id": "github-copilot", "term": "GitHub Copilot", "aliases": ["Copilot"], "lecture": 1, "category": "AIコーディング", "definition": "GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。", "related": ["code-completion", "coding-agent"]}, {"id": "code-completion", "term": "コード補完", "aliases": ["インライン補完", "inline suggestion"], "lecture": 1, "category": "AIコーディング", "definition": "書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。", "related": ["github-copilot"]}, {"id": "coding-agent", "term": "コーディングエージェント", "aliases": ["Coding Agent", "AIコーディングエージェント"], "lecture": 1, "category": "AIコーディング", "definition": "ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。", "related": ["ai-agent", "harness"]}, {"id": "pretraining", "term": "事前学習", "aliases": ["pre-training", "プレトレーニング"], "lecture": 2, "category": "ファインチューニング", "definition": "大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。", "related": ["fine-tuning", "llm"]}, {"id": "fine-tuning", "term": "ファインチューニング", "aliases": ["fine-tuning", "追加学習", "FT"], "lecture": 2, "category": "ファインチューニング", "definition": "事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。", "related": ["sft", "peft", "lora"]}, {"id": "sft", "term": "教師ありファインチューニング（SFT）", "aliases": ["SFT", "Supervised Fine-Tuning", "指示チューニング"], "lecture": 2, "category": "ファインチューニング", "definition": "入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。", "related": ["fine-tuning", "lora"]}, {"id": "peft", "term": "PEFT", "aliases": ["Parameter-Efficient Fine-Tuning", "パラメータ効率の良いファインチューニング"], "lecture": 2, "category": "ファインチューニング", "definition": "モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。", "related": ["lora", "fine-tuning"]}, {"id": "lora", "term": "LoRA", "aliases": ["Low-Rank Adaptation"], "lecture": 2, "category": "ファインチューニング", "definition": "事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。", "related": ["peft", "lora-rank", "qlora"]}, {"id": "lora-rank", "term": "ランク（LoRAのr）", "aliases": ["rank", "r"], "lecture": 2, "category": "ファインチューニング", "definition": "LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。", "related": ["lora"]}, {"id": "qlora", "term": "QLoRA", "aliases": ["Quantized LoRA"], "lecture": 2, "category": "ファインチューニング", "definition": "ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。", "related": ["lora", "quantization"]}, {"id": "quantization", "term": "量子化", "aliases": ["quantization"], "lecture": 2, "category": "ファインチューニング", "definition": "モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。", "related": ["qlora"]}, {"id": "learning-rate", "term": "学習率", "aliases": ["learning rate", "lr"], "lecture": 2, "category": "ファインチューニング", "definition": "1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。", "related": ["epoch", "overfitting"]}, {"id": "epoch", "term": "エポック", "aliases": ["epoch"], "lecture": 2, "category": "ファインチューニング", "definition": "学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。", "related": ["overfitting", "learning-rate"]}, {"id": "overfitting", "term": "過学習", "aliases": ["overfitting", "オーバーフィッティング"], "lecture": 2, "category": "ファインチューニング", "definition": "学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。", "related": ["epoch", "evaluation-loop"]}, {"id": "huggingface-hub", "term": "Hugging Face Hub", "aliases": ["Hugging Face", "HF Hub"], "lecture": 2, "category": "ファインチューニング", "definition": "公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。", "related": ["lora", "pretraining"]}, {"id": "ai-agent", "term": "AIエージェント", "aliases": ["エージェント", "agent"], "lecture": 3, "category": "エージェント", "definition": "LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。", "related": ["agent-loop", "tool", "harness"]}, {"id": "agent-loop", "term": "エージェントループ", "aliases": ["agent loop", "ツール呼び出しループ"], "lecture": 3, "category": "エージェント", "definition": "モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。", "related": ["ai-agent", "tool", "rate-limit"]}, {"id": "tool", "term": "ツール（Tool use / Function calling）", "aliases": ["ツール呼び出し", "Function calling", "Tool use", "ツール"], "lecture": 3, "category": "エージェント", "definition": "モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。", "related": ["tool-description", "agent-loop"]}, {"id": "tool-description", "term": "ツールの説明文", "aliases": ["docstring", "ツールスキーマ", "tool schema"], "lecture": 3, "category": "エージェント", "definition": "ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。", "related": ["tool", "pydantic-ai"]}, {"id": "structured-output", "term": "構造化出力", "aliases": ["structured output", "JSON出力"], "lecture": 3, "category": "エージェント", "definition": "モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。", "related": ["pydantic-ai"]}, {"id": "harness", "term": "ハーネス", "aliases": ["agent harness", "エージェントハーネス"], "lecture": 3, "category": "エージェント", "definition": "LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。", "related": ["coding-agent", "pi"]}, {"id": "pi", "term": "Pi", "aliases": ["Pi coding agent", "pi agent"], "lecture": 3, "category": "エージェント", "definition": "ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。", "related": ["harness", "coding-agent"]}, {"id": "pydantic-ai", "term": "Pydantic AI", "aliases": ["pydantic-ai", "PydanticAI"], "lecture": 3, "category": "エージェント", "definition": "Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。", "related": ["test-model", "structured-output", "adk"]}, {"id": "adk", "term": "ADK（Agent Development Kit）", "aliases": ["ADK", "Agent Development Kit", "google-adk"], "lecture": 3, "category": "エージェント", "definition": "Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。", "related": ["pydantic-ai", "ai-agent"]}, {"id": "mcp", "term": "MCP（Model Context Protocol）", "aliases": ["MCP", "Model Context Protocol"], "lecture": 3, "category": "エージェント", "definition": "AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。", "related": ["tool", "ai-agent", "mcp-server", "mcp-client"]}, {"id": "guardrail", "term": "ガードレール", "aliases": ["guardrail"], "lecture": 3, "category": "エージェント", "definition": "エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。", "related": ["human-in-the-loop", "system-prompt"]}, {"id": "human-in-the-loop", "term": "Human-in-the-loop", "aliases": ["HITL", "人間による承認", "確認を挟む"], "lecture": 3, "category": "エージェント", "definition": "エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。", "related": ["guardrail", "tool"]}, {"id": "test-model", "term": "TestModel", "aliases": ["テスト用モデル"], "lecture": 3, "category": "エージェント", "definition": "Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。", "related": ["pydantic-ai", "rate-limit"]}, {"id": "homework-levels", "term": "宿題の3レベル", "aliases": ["3レベル", "宿題レベル"], "lecture": 1, "category": "講座", "definition": "AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。", "related": ["answer-md"]}, {"id": "answer-md", "term": "answer.md", "aliases": ["解答ファイル"], "lecture": 1, "category": "講座", "definition": "AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。", "related": ["pull-request", "homework-levels"]}, {"id": "lec-start-tag", "term": "lecNN-start タグ", "aliases": ["開始タグ", "lec03-start"], "lecture": 1, "category": "講座", "definition": "AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。", "related": ["branch"]}, {"id": "default-model", "term": "既定モデル", "aliases": ["デフォルトモデル", "gemini-3.5-flash-lite"], "lecture": 1, "category": "講座", "definition": "AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。", "related": ["rate-limit", "api-key"]}, {"id": "external-tool", "term": "外部ツール", "aliases": ["external tool", "外部サービス"], "lecture": 5, "category": "MCP", "definition": "Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。", "related": ["tool", "mcp", "mcp-server"]}, {"id": "mcp-server", "term": "MCPサーバー", "aliases": ["MCP server"], "lecture": 5, "category": "MCP", "definition": "MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。", "related": ["mcp", "mcp-client", "mcp-transport", "fetch-server"]}, {"id": "mcp-client", "term": "MCPクライアント", "aliases": ["MCP client", "MCPホスト"], "lecture": 5, "category": "MCP", "definition": "MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。", "related": ["mcp", "mcp-server", "mcp-config"]}, {"id": "mcp-transport", "term": "トランスポート（stdio・Streamable HTTP）", "aliases": ["stdio", "Streamable HTTP", "transport", "MCPのトランスポート"], "lecture": 5, "category": "MCP", "definition": "MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。", "related": ["mcp-server", "mcp-client"]}, {"id": "mcp-config", "term": "mcpServers設定", "aliases": ["mcpServers", "MCPの設定ファイル", "mcp_servers.json", "mcp.json"], "lecture": 5, "category": "MCP", "definition": "使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。", "related": ["mcp-client", "tool-namespace"]}, {"id": "tool-namespace", "term": "ツールの名前空間", "aliases": ["ツール名のプレフィックス", "prefix", "名前の衝突"], "lecture": 5, "category": "MCP", "definition": "複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。", "related": ["mcp-config", "tool-description", "rate-limit"]}, {"id": "fetch-server", "term": "fetchサーバー", "aliases": ["mcp-server-fetch", "fetch", "Web取得ツール"], "lecture": 5, "category": "MCP", "definition": "指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。", "related": ["mcp-server", "external-tool", "guardrail"]}], "train": [{"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「AIエンジニアリング」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "AIエンジニアリングについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「評価と改善のサイクル」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "評価と改善のサイクルについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「LLM（大規模言語モデル）」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "LLM（大規模言語モデル）について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「トークン」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "トークンについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「コンテキストウィンドウ」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "コンテキストウィンドウについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「プロンプト」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "プロンプトについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「システムプロンプト」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "システムプロンプトについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「temperature」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "temperatureについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ハルシネーション」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ハルシネーションについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「API」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "APIについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「APIキー」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "APIキーについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「環境変数」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "環境変数について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「レート制限」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "レート制限について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「GitHub Codespaces」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "GitHub Codespacesについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「devcontainer」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "devcontainerについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「Codespaces secret」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Codespaces secretについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「fork」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "forkについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ブランチ」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ブランチについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「プルリクエスト（PR）」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "プルリクエスト（PR）について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「GitHub Copilot」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "GitHub Copilotについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「コード補完」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "コード補完について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「コーディングエージェント」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "コーディングエージェントについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「事前学習」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "事前学習について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ファインチューニング」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ファインチューニングについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「教師ありファインチューニング（SFT）」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "教師ありファインチューニング（SFT）について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「PEFT」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "PEFTについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「LoRA」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "LoRAについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ランク（LoRAのr）」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ランク（LoRAのr）について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「QLoRA」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "QLoRAについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「量子化」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "量子化について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「学習率」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "学習率について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「エポック」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "エポックについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「過学習」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "過学習について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「Hugging Face Hub」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Hugging Face Hubについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「AIエージェント」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "AIエージェントについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「エージェントループ」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "エージェントループについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ツール（Tool use / Function calling）」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ツール（Tool use / Function calling）について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ツールの説明文」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ツールの説明文について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「構造化出力」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "構造化出力について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ハーネス」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ハーネスについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「Pi」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Piについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「Pydantic AI」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Pydantic AIについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ADK（Agent Development Kit）」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ADK（Agent Development Kit）について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「MCP（Model Context Protocol）」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "MCP（Model Context Protocol）について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ガードレール」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ガードレールについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「Human-in-the-loop」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Human-in-the-loopについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「TestModel」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "TestModelについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「宿題の3レベル」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "宿題の3レベルについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「answer.md」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "answer.mdについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「lecNN-start タグ」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "lecNN-start タグについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「既定モデル」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "既定モデルについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「外部ツール」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "外部ツールについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「MCPサーバー」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "MCPサーバーについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「MCPクライアント」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "MCPクライアントについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「トランスポート（stdio・Streamable HTTP）」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "トランスポート（stdio・Streamable HTTP）について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「mcpServers設定」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "mcpServers設定について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ツールの名前空間」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ツールの名前空間について簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「fetchサーバー」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "fetchサーバーについて簡単に説明してください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「ドロップアウト」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"用語集に記載がありません。確認が必要です。\\", \\"needs_review\\": true}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「バッチ正規化」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"用語集に記載がありません。確認が必要です。\\", \\"needs_review\\": true}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「交差エントロピー」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"用語集に記載がありません。確認が必要です。\\", \\"needs_review\\": true}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "「勾配クリッピング」とは何ですか？"}, {"role": "assistant", "content": "{\\"answer\\": \\"用語集に記載がありません。確認が必要です。\\", \\"needs_review\\": true}"}]}], "valid": [{"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "AIエンジニアリングの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "評価と改善のサイクルの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"作る→測る→直すを繰り返す開発の進め方。AIアプリは出力が確率的に揺れるため、感覚ではなく評価データや指標で良し悪しを判断し、その結果をもとにプロンプト・データ・モデル・ツールを改善する。本講座では学習支援エージェントを題材に、回をまたいでこのサイクルを回す。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "LLM（大規模言語モデル）の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"大量のテキストで学習し、入力に続くトークンを予測することで文章を生成するモデル。質問応答、要約、コード生成など多くのタスクを、指示（プロンプト）を変えるだけでこなせる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "トークンの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "コンテキストウィンドウの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "プロンプトの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMに渡す入力文。質問だけでなく、役割、制約、出力形式、例などを含めることで出力を方向づけられる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "システムプロンプトの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"利用者の質問とは別に、モデルの振る舞い（役割、口調、守るべきルールなど）を指定する入力。Pydantic AIでは Agent の instructions 引数で与える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "temperatureの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"次のトークンを選ぶときのランダムさを調整する設定値。低いほど出力が安定し、高いほど多様になる。同じ質問でも回答が変わる理由の一つ。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ハルシネーションの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "APIの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"プログラムから別のサービスの機能を呼び出すための窓口。本講座ではGemini APIをPythonから呼び出してLLMを使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "APIキーの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"APIを呼び出す人を識別するための秘密の文字列。漏れると他人に利用枠を使われるため、コードに直接書かない、コミットしない、チャットに貼らない。本講座ではCodespaces secretに登録し、環境変数 GEMINI_API_KEY として読み込む。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "環境変数の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"プログラムの外側（OSや実行環境）で設定し、プログラムから名前で読み出せる値。APIキーのような秘密情報や、環境ごとに変わる設定をコードから切り離すために使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "レート制限の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"一定時間あたりに呼び出せるリクエスト数やトークン数の上限。超えるとHTTP 429などのエラーが返る。無料枠は特に制限が厳しく、エージェントは1つの質問で複数回LLMを呼ぶため上限に達しやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "GitHub Codespacesの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubが提供するクラウド上の開発環境。ブラウザからVS Codeを開き、repoに書かれた設定どおりの環境をすぐに使える。個人アカウントには毎月の無料枠があり、使わないときは停止しないと枠を消費し続ける。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "devcontainerの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"開発環境をコンテナとして定義する仕組み。.devcontainer/devcontainer.json にPythonのバージョンや拡張機能、起動時のコマンドを書いておくと、誰がCodespacesを開いても同じ環境になる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Codespaces secretの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubのアカウント設定に登録しておく秘密の値。指定したrepoのCodespacesを起動すると環境変数として渡される。登録時に使うrepo（自分のfork）を選ぶ必要がある。起動中に登録した場合はCodespacesの再起動が必要。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "forkの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"他人のrepoを自分のアカウントに丸ごとコピーすること。本講座では講座repoをforkし、自分のfork上で作業・提出する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ブランチの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"作業の流れを分けるための仕組み。main を壊さずに変更を試し、まとまったらPRで main に取り込む。本講座では回ごとに lec01 のような作業ブランチを作る。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "プルリクエスト（PR）の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"あるブランチの変更を別のブランチへ取り込むよう依頼する仕組み。差分の確認、コメント、自動テストの結果表示ができる。本講座の提出は、自分のfork内で作業ブランチから main へのPRとして行う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "GitHub Copilotの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"GitHubのAIコーディング支援。エディタでのコード補完、チャットでの質問、エージェントモードでの複数ファイル編集などができる。無料のCopilot Freeは補完の回数とチャットの利用量に上限がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "コード補完の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"書きかけのコードやコメントから、続きのコードをAIが提案する機能。Tabキーなどで受け入れる。提案は正しいとは限らないので、読んで理解してから採用する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "コーディングエージェントの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"ファイルの読み書きやコマンド実行をツールとして使い、指示されたプログラミング作業を複数ステップで自律的に進めるAI。Claude Code、Copilotのエージェントモード、Piなどがある。中身はLLM＋ループ＋ツールというAIエージェントの構造そのもの。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "事前学習の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"大量の汎用テキストで、次のトークンを予測するようにモデルを学習させる最初の段階。言語の知識や一般的な常識の多くはここで身につく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ファインチューニングの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"事前学習済みモデルを、特定のタスクや形式に合ったデータで追加学習させること。出力の形式や口調、特定ドメインの振る舞いを身につけさせるのに向く。最新の事実を覚えさせる用途には、ツールやRAGで外部データを参照させるほうが向くことが多い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "教師ありファインチューニング（SFT）の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"入力と望ましい出力のペア（例：質問と模範回答）を使ってモデルを追加学習させる方法。指示に従う振る舞いを身につけさせる代表的な手法。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "PEFTの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの全パラメータではなく、ごく一部や追加した小さなパラメータだけを学習する手法の総称。必要なGPUメモリと保存容量を大きく減らせる。LoRAが代表例。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "LoRAの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ランク（LoRAのr）の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LoRAで使う低ランク行列の大きさを決めるハイパーパラメータ。大きいほど表現力が上がるが、学習するパラメータとメモリも増える。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "QLoRAの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "量子化の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの重みを少ないビット数（例：16ビットから4ビット）で表し、メモリ使用量や計算量を減らす技術。精度は少し下がることがある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "学習率の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "エポックの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "過学習の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Hugging Face Hubの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"公開されたモデルやデータセットを共有・ダウンロードできるプラットフォーム。オープンウェイトのモデルを入手したり、LoRAアダプタを公開したりするのに使われる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "AIエージェントの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "エージェントループの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルに入力を渡す→モデルがツール呼び出しを返したら実行する→結果をモデルに戻す、をモデルが最終回答を返すまで繰り返す処理。1つの質問でLLMを複数回呼ぶため、時間・料金・レート制限への影響が大きい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ツール（Tool use / Function calling）の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルが呼び出せるように用意した関数。モデルはツールの名前・説明・引数の型を見て、必要なときに引数を決めて呼び出しを依頼し、実行はプログラム側が行う。検索、計算、ファイル操作、外部APIの呼び出しなどに使う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ツールの説明文の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"ツールの用途や引数の意味をモデルに伝える文章。Pydantic AIでは関数のdocstringと型ヒントから自動で作られる。モデルはこれを読んでツールを使うか、どの引数で呼ぶかを決めるので、書き方次第で挙動が大きく変わる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "構造化出力の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"モデルの出力を、決めた型やJSONスキーマに沿った形で受け取る仕組み。プログラムで後処理しやすくなる。Pydantic AIでは output_type にPydanticモデルを指定する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ハーネスの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"LLMの周りを、ループ・ツール・画面（UI）などで包み、実際に作業できるようにするプログラム。Claude CodeやPiはハーネスの例。Piは組み込みツールをread・write・edit・bashの4つに絞った薄いハーネスとして知られる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Piの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"ターミナルで動くオープンソース（MITライセンス）のコーディングエージェント。最小限の機能とシステムプロンプトで構成し、利用者が拡張して使う設計。2026年10月に1.0が公開された。npmのパッケージ名は @earendil-works/pi-coding-agent。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Pydantic AIの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"Pythonでエージェントを作るためのフレームワーク。Pydanticの型を使ってツールの引数や出力を定義でき、GeminiやOpenAIなど多くのモデルを同じ書き方で扱える。テスト用のモデルでAPIを呼ばずに動作確認できる。本講座第3回の本線。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ADK（Agent Development Kit）の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"Googleが公開しているオープンソースのエージェント開発フレームワーク。Geminiとの統合が深く、adk web でツール呼び出しの流れを画面で確認できる。2.0で1.x系から破壊的変更が入っている。本講座では宿題の発展レベルで扱う。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "MCP（Model Context Protocol）の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIアプリと外部のツールやデータソースをつなぐためのオープンな標準プロトコル。MCPサーバーとして公開したツールは、対応する様々なエージェントやエディタ（MCPクライアント）から共通の方法で使える。第5回で、用語集の検索をMCPサーバーにして体験する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ガードレールの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"エージェントが望ましくない行動や出力をしないように設ける制約や検査。指示文でのルール、ツールの権限を絞る、出力をチェックする、重要な操作の前に人の確認を挟む、などの方法がある。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "Human-in-the-loopの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"エージェントの処理の途中に人の確認や承認を組み込むこと。データの書き込みや送信など、取り消しにくい操作の前に使う。Pydantic AIでは requires_approval=True を指定したツールで実現できる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "TestModelの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"Pydantic AIに含まれるテスト用の偽モデル。実際のLLMを呼ばずに、ツールを呼び出してダミーの回答を返す。APIキーや利用枠を使わずに、エージェントの配線（ツールが登録され、呼ばれ、結果が返るか）をテストできる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "宿題の3レベルの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の宿題の構成。「基礎（講義内容の確認）」「実装（自分でコードを書く）」「発展（調査や試行）」の3段階があり、どれか1つに取り組めばその回の宿題は完了。原則として採点はしない。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "answer.mdの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026で各回の演習・宿題の結果を書くファイル。lectures/NN/answer.md にあり、取り組んだレベル・やったこと・結果・気づきを記入して、作業ブランチから自分のforkの main へPRを出して提出する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "lecNN-start タグの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の講座repoで、各回の演習開始時点のコードに付けているGitタグ（例：lec03-start）。前の回が終わっていなくても、このタグからブランチを作ればその回の演習を始められる。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "既定モデルの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"AIE2026の演習と宿題で標準に使うモデルは gemini-3.5-flash-lite。無料枠のレート制限と講座運営の安定性を考えて選んでいる。環境変数 AIE_MODEL で他のモデルに切り替えてよく、自分で契約したAPIを使ってもよい。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "外部ツールの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"Webページの取得、ファイル操作、データベース、他社のAPIなど、エージェントのプログラムの外にある機能をツールとして使うこと。第3回の関数ツールは同じプロセスの中で動くが、外部ツールはプロセスや機械をまたいで呼び出す。MCPはそのための共通の規約。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "MCPサーバーの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPでツール（やリソース、プロンプト）を公開する側のプログラム。公開するのはツールの名前・説明文・引数のスキーマで、クライアントの要求に応じてツールを実行し結果を返す。本講座では用語集の検索をMCPサーバーにする（app/mcp_server.py）。公開されている既製のサーバー（fetch、filesystem、GitHubなど）も多い。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "MCPクライアントの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPサーバーに接続し、公開されたツールをモデルに渡して、モデルの指示でツールを呼び出す側。エージェントのプログラム（Pydantic AIの MCPToolset）や、Copilot Chat・Claude Desktop・Cursorなどのエディタやアプリがクライアントになる。同じサーバーをどのクライアントからも使えるのがMCPの利点。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "トランスポート（stdio・Streamable HTTP）の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"MCPのクライアントとサーバーがやりとりする経路。stdioでは、クライアントがサーバーをサブプロセスとして起動し、標準入出力でメッセージを交換する（手軽で、同じ機械の中で使う）。Streamable HTTPでは、サーバーがURLで待ち受け、別のプロセスや別の機械から接続できる（公開範囲と認証を考える必要がある）。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "mcpServers設定の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"使うMCPサーバーを列挙する設定ファイル。サーバーごとに名前と、起動コマンド（stdio）または接続先URL（HTTP）を書く。Claude Desktop、Cursor、Pydantic AIなどが同じ mcpServers 形式を読む（VS Code / Copilot Chatは servers 形式）。本講座では app/mcp_servers.json。設定を数行足すだけでツールが増えるのがMCPの体験の核心。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "ツールの名前空間の意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"複数のMCPサーバーを使うとき、同じ名前のツールを区別するために「サーバー名_ツール名」のように接頭辞を付けること。本講座の app/agent.py は設定ファイルのサーバー名を自動で付ける（例: glossary_search_glossary。Pydantic AIの load_mcp_toolsets も同じ）。ツールが増えるほどモデルに渡す説明文も増え、呼び分けの精度とトークン数に影響する。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "fetchサーバーの意味を一文で教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"指定したURLのページを取得し、本文をMarkdownにして返す既製のMCPサーバー（Python製、uvx mcp-server-fetch で起動）。本講座では「設定ファイルに書くだけで外部ツールが増える」例として使う。モデルが選んだURLに実際にアクセスするので、どこへ何を送るかは利用者が把握しておく。\\", \\"needs_review\\": false}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "蒸留の意味を教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"用語集に記載がありません。確認が必要です。\\", \\"needs_review\\": true}"}]}, {"messages": [{"role": "system", "content": "あなたは講座の用語集の回答係です。学習対象の用語集に基づいて回答してください。出力は JSON 1 行で、answer（文字列）と needs_review（真偽値）の2キーだけにしてください。用語集にない語には「用語集に記載がありません。確認が必要です。」と答え、needs_review を true にしてください。"}, {"role": "user", "content": "自己注意機構の意味を教えてください。"}, {"role": "assistant", "content": "{\\"answer\\": \\"用語集に記載がありません。確認が必要です。\\", \\"needs_review\\": true}"}]}], "test": [{"question": "用語集の「LoRA」の定義を教えてください。", "category": "用語あり", "term_id": "lora"}, {"question": "元の重みを変えずに、低ランクの行列だけを学習する方法を説明してください。", "category": "言い換え", "term_id": "lora"}, {"question": "用語集の「QLoRA」の定義を教えてください。", "category": "用語あり", "term_id": "qlora"}, {"question": "4ビットに圧縮した重みの上にLoRAを付ける方法は何ですか？", "category": "言い換え", "term_id": "qlora"}, {"question": "用語集の「過学習」の定義を教えてください。", "category": "用語あり", "term_id": "overfitting"}, {"question": "訓練用の例だけに適合して、初めての入力で成績が落ちる状態とは？", "category": "言い換え", "term_id": "overfitting"}, {"question": "用語集の「学習率」の定義を教えてください。", "category": "用語あり", "term_id": "learning-rate"}, {"question": "learning rateが大きすぎる場合と小さすぎる場合の違いを教えてください。", "category": "言い換え", "term_id": "learning-rate"}, {"question": "用語集の「エポック」の定義を教えてください。", "category": "用語あり", "term_id": "epoch"}, {"question": "epochは学習データを何回使ったかとどう関係しますか？", "category": "言い換え", "term_id": "epoch"}, {"question": "用語集の「コンテキストウィンドウ」の定義を教えてください。", "category": "用語あり", "term_id": "context-window"}, {"question": "context windowに会話履歴も収める必要があるのはなぜですか？", "category": "言い換え", "term_id": "context-window"}, {"question": "用語集の「BERT」の定義を教えてください。", "category": "用語集外", "term_id": null}, {"question": "用語集の「ベクトルデータベース」の定義を教えてください。", "category": "用語集外", "term_id": null}, {"question": "用語集の「DPO」の定義を教えてください。", "category": "用語集外", "term_id": null}, {"question": "用語集の「RLHF」の定義を教えてください。", "category": "用語集外", "term_id": null}, {"question": "用語集の「MoE」の定義を教えてください。", "category": "用語集外", "term_id": null}, {"question": "用語集の「破滅的忘却」の定義を教えてください。", "category": "用語集外", "term_id": null}], "gold": {"用語集の「LoRA」の定義を教えてください。": {"answer": "事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。", "needs_review": false, "must_include_groups": [["固定"], ["小さな"]]}, "元の重みを変えずに、低ランクの行列だけを学習する方法を説明してください。": {"answer": "事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。", "needs_review": false, "must_include_groups": [["固定"], ["行列"], ["学習"]]}, "用語集の「QLoRA」の定義を教えてください。": {"answer": "ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。", "needs_review": false, "must_include_groups": [["4ビット"], ["LoRA"]]}, "4ビットに圧縮した重みの上にLoRAを付ける方法は何ですか？": {"answer": "ベースモデルの重みを4ビットなどに量子化してメモリを節約したうえで、LoRAで学習する手法。より大きなモデルを小さなGPUで扱える。本講座の第2回演習ではモデルがGPUに載るためLoRAで学習し、4ビットで読み込んだときのメモリと出力の違いを発展で比べる。", "needs_review": false, "must_include_groups": [["4bit", "4ビット", "4 bit"], ["LoRA"]]}, "用語集の「過学習」の定義を教えてください。": {"answer": "学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。", "needs_review": false, "must_include_groups": [["学習データ"], ["未知"]]}, "訓練用の例だけに適合して、初めての入力で成績が落ちる状態とは？": {"answer": "学習データにだけ過度に適合し、未知の入力に対する性能が落ちる状態。学習に使わない検証データで性能を測って気づく。", "needs_review": false, "must_include_groups": [["学習データ", "訓練データ"], ["未知", "初めて", "汎化"]]}, "用語集の「学習率」の定義を教えてください。": {"answer": "1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。", "needs_review": false, "must_include_groups": [["更新"], ["パラメータ"]]}, "learning rateが大きすぎる場合と小さすぎる場合の違いを教えてください。": {"answer": "1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。", "needs_review": false, "must_include_groups": [["不安定"], ["遅い", "遅く"]]}, "用語集の「エポック」の定義を教えてください。": {"answer": "学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。", "needs_review": false, "must_include_groups": [["1周"]]}, "epochは学習データを何回使ったかとどう関係しますか？": {"answer": "学習データ全体を1周学習することを1エポックと数える。回数を増やしすぎると過学習しやすい。", "needs_review": false, "must_include_groups": [["1周", "一周", "1 周"], ["学習"]]}, "用語集の「コンテキストウィンドウ」の定義を教えてください。": {"answer": "LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。", "needs_review": false, "must_include_groups": [["入力"], ["出力"], ["上限"]]}, "context windowに会話履歴も収める必要があるのはなぜですか？": {"answer": "LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。", "needs_review": false, "must_include_groups": [["入力"], ["出力"], ["上限"]]}, "用語集の「BERT」の定義を教えてください。": {"answer": "用語集に記載がありません。確認が必要です。", "needs_review": true, "must_include_groups": [["確認"]]}, "用語集の「ベクトルデータベース」の定義を教えてください。": {"answer": "用語集に記載がありません。確認が必要です。", "needs_review": true, "must_include_groups": [["確認"]]}, "用語集の「DPO」の定義を教えてください。": {"answer": "用語集に記載がありません。確認が必要です。", "needs_review": true, "must_include_groups": [["確認"]]}, "用語集の「RLHF」の定義を教えてください。": {"answer": "用語集に記載がありません。確認が必要です。", "needs_review": true, "must_include_groups": [["確認"]]}, "用語集の「MoE」の定義を教えてください。": {"answer": "用語集に記載がありません。確認が必要です。", "needs_review": true, "must_include_groups": [["確認"]]}, "用語集の「破滅的忘却」の定義を教えてください。": {"answer": "用語集に記載がありません。確認が必要です。", "needs_review": true, "must_include_groups": [["確認"]]}}, "manifest": {"source_repository": "https://github.com/matsuolab/lecture-ai-engineering-2026", "source_commit": "c17c00e940143e5af03b228d2aac61c5af492bf9", "glossary_terms": 58, "raw_examples": 116, "train": 120, "valid": 60, "test": 18, "local_training_executed": false, "generation": "講座スクリプトを変更せずPython標準ライブラリで実行（uv run と同じmain処理）", "sha256": {"glossary_sft.jsonl": "af6ac820e0331d8d9af376e3b4cccff1b31f2f0f8bb462839752e3282bbd1081", "gold.json": "02819036cf7ee4ff1f8cbd6cd15e402446ebe97b1aba03da7b522ef28e292b4a", "test.jsonl": "cf93dc41fef804723b96086148e8483a62d85c66a24bf3de4a9c7a4769403b34", "train.jsonl": "cebeb3729e7c10d81d2ec89ace912624af3d411806080406be8de3b434f04690", "valid.jsonl": "0472467d498e2498b98177808bfb18413fe2f74366d5de2790dd1ee835ddd1e8"}}}')

TERMS = PAYLOAD['terms']; test = PAYLOAD['test']; gold = PAYLOAD['gold']
SYSTEM = PAYLOAD['train'][0]['messages'][0]['content']
CONFIG = dict(EXERCISE_RESULT['config'])
CONFIG.update(num_train_epochs=3, max_new_tokens=256, work_dir='/content/lesson02_advanced', drive_dir='/content/drive/MyDrive/aie2026/lesson02_advanced')
WORK = Path(CONFIG['work_dir']); WORK.mkdir(exist_ok=True)
USE_DRIVE = False
for key in ['train','valid','test']:
    (WORK/f'{key}.jsonl').write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in PAYLOAD[key]),encoding='utf-8')
(WORK/'gold.json').write_text(json.dumps(gold,ensure_ascii=False,indent=2),encoding='utf-8')
print({k:len(PAYLOAD[k]) for k in ['train','valid','test']})


In [ ]:
"""発展課題の採点・用語検索。標準ライブラリだけで事前確認できる。"""
import json
import re
import unicodedata


def norm(s):
    return re.sub(r'\s+', '', unicodedata.normalize('NFKC', s)).casefold()


def retrieve(question, terms):
    """用語名または別名の最長一致。正解ファイルは参照しない。"""
    q = norm(question)
    candidates = []
    for t in terms:
        hits = []
        for a in [t['term']] + t['aliases']:
            name = norm(a)
            if not name:
                continue
            # r や FT のような短い英字別名を、別の英単語の一部として拾わない
            matched = (re.search(r'(?<![a-z0-9_])' + re.escape(name) + r'(?![a-z0-9_])', q)
                       if name.isascii() else name in q)
            if matched:
                hits.append(len(name))
        if hits:
            candidates.append((max(hits), t['id'], t))
    return sorted(candidates, key=lambda x: (-x[0], x[1]))[0][2] if candidates else None


def score(raw, gold):
    try:
        obj = json.loads(raw)
    except (ValueError, TypeError):
        # 原ノートブック同様、前後に余分な文章がある場合は内容だけ別判定
        m = re.search(r'\{.*\}', raw, re.S)
        try:
            obj = json.loads(m.group()) if m else None
        except ValueError:
            obj = None
        strict = False
    else:
        strict = True
    typed = (isinstance(obj, dict) and isinstance(obj.get('answer'), str)
             and type(obj.get('needs_review')) is bool)
    format_ok = bool(strict and typed and set(obj) == {'answer', 'needs_review'})
    content_ok = bool(typed and obj['needs_review'] == gold['needs_review'] and all(
        any(norm(w) in norm(obj['answer']) for w in alternatives)
        for alternatives in gold['must_include_groups']))
    return {'format_ok': format_ok, 'content_ok': content_ok,
            'both_ok': format_ok and content_ok,
            'classification': ('format_error' if not format_ok else
                               'needs_review' if obj['needs_review'] else 'asserted')}


def summarize(rows, label):
    def rates(sub):
        return {'n': len(sub), **{k.replace('_ok', '_rate'): sum(r[k] for r in sub)/len(sub)
                                 for k in ['format_ok', 'content_ok', 'both_ok']}}
    result = {'label': label, **rates(rows)}
    result['categories'] = {c: rates([r for r in rows if r['category'] == c])
                            for c in sorted({r['category'] for r in rows})}
    return result


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)


# 学習データが黙って切り詰められないことを確認
lengths = [len(tok.apply_chat_template(x['messages'],tokenize=True,enable_thinking=False)) for x in PAYLOAD['train']+PAYLOAD['valid']]
print('学習・検証例の最大トークン長:', max(lengths))
assert max(lengths) <= CONFIG['max_seq_len'], 'max_seq_lenを超えています。設定を見直してセッションを再起動してください。'

def build_messages(example, retrieval=False):
    user = example['question']
    if retrieval:
        hit = retrieve(example['question'], TERMS)
        context = hit['definition'] if hit else '検索結果なし。用語集の記載を確認できませんでした。'
        user = f"用語集の検索結果（資料）:\n{context}\n\n質問: {user}"
    return [{'role':'system','content':SYSTEM},{'role':'user','content':user}]

def generate(model, example, retrieval=False):
    prompt=tok.apply_chat_template(build_messages(example,retrieval),tokenize=False,add_generation_prompt=True,enable_thinking=False)
    inputs=tok(prompt,return_tensors='pt').to(model.device)
    with torch.no_grad():
        output=model.generate(**inputs,max_new_tokens=CONFIG['max_new_tokens'],pad_token_id=tok.pad_token_id,**CONFIG['gen'])
    return tok.decode(output[0][inputs['input_ids'].shape[1]:],skip_special_tokens=True).strip()

def evaluate_advanced(model,label,retrieval=False):
    rows=[]; start=time.perf_counter()
    for ex in test:
        raw=generate(model,ex,retrieval)
        rows.append({**ex,'raw':raw,**score(raw,gold[ex['question']])})
    summary=summarize(rows,label);summary['seconds']=round(time.perf_counter()-start,2)
    (WORK/f'eval_{label}.json').write_text(json.dumps({'summary':summary,'rows':rows},ensure_ascii=False,indent=2),encoding='utf-8')
    display(pd.DataFrame(rows)); print(summary)
    return summary,rows
before_summary,before_rows=evaluate_advanced(model,'before')
rag_summary,rag_rows=evaluate_advanced(model,'retrieval_base',retrieval=True)
# 純粋な用語名・別名検索の結果も別に保存。RAGのモデル出力とは区別する
lookup_rows=[]
for ex in test:
    hit=retrieve(ex['question'],TERMS)
    raw=json.dumps({'answer':hit['definition'] if hit else '用語集に記載がありません。確認が必要です。','needs_review':hit is None},ensure_ascii=False)
    lookup_rows.append({**ex,'raw':raw,**score(raw,gold[ex['question']])})
lookup_summary=summarize(lookup_rows,'lookup_only')
(WORK/'eval_lookup_only.json').write_text(json.dumps({'summary':lookup_summary,'rows':lookup_rows},ensure_ascii=False,indent=2),encoding='utf-8')


## 用語集でLoRA学習
講座スクリプトの116件をもとに、教師回答をJSONに揃え、用語集外への確認要求4件を追加しました。
学習時の入力には用語集の定義を入れません。RAGの比較時だけ検索で得た定義を入力に入れます。
検証用は別の質問テンプレートです。同じ用語を扱うので、未知の概念への汎化を測る分割ではありません。


In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list(PAYLOAD['train'])
valid_ds = Dataset.from_list(PAYLOAD['valid'])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
training_start = time.perf_counter()
train_output = trainer.train()
training_seconds = time.perf_counter()-training_start
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

In [ ]:

after_summary,after_rows=evaluate_advanced(tuned,'after')
regressed=[b['question'] for b,a in zip(before_rows,after_rows) if b['both_ok'] and not a['both_ok']]
loss_logs=[x['loss'] for x in trainer.state.log_history if 'loss' in x]
result={'config':CONFIG,'versions':{m.__name__:m.__version__ for m in [transformers,peft,trl,accelerate,datasets,torch]},
        'method':'LoRA（量子化なし）','provenance':PAYLOAD['manifest'],
        'before':before_summary,'after':after_summary,'retrieval_base':rag_summary,'lookup_only':lookup_summary,
        'regressed':regressed,'training_seconds':training_seconds,'training_steps':trainer.state.global_step,
        'train_loss_mean':train_output.training_loss,'last_logged_loss':loss_logs[-1] if loss_logs else None,
        'executed_training_myself':True}
(WORK/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([before_summary,after_summary,rag_summary,lookup_summary])[['label','n','format_rate','content_rate','both_rate']])
print('退行した質問:',regressed)


## 解答文と実験結果を保存する
次のセルは実測値だけを使って `answer.md` を作ります。出力原文・正解を読み、採点の妥当性と失敗理由を確認してから提出してください。
自動採点は必須語句による近似です。手動で判定を修正した場合は、その理由もanswer.mdに追記します。


In [ ]:
# ノートブックの実測変数から提出文を作る。単独実行用ではない。
def pct(x): return f'{x:.1%}'
def table(summaries):
    lines=['| 条件 | 問数 | 形式遵守率 | 内容正解率 | 両方正解 |','|---|---:|---:|---:|---:|']
    for s in summaries:
        lines.append(f"| {s['label']} | {s['n']} | {pct(s['format_rate'])} | {pct(s['content_rate'])} | {pct(s['both_rate'])} |")
    return '\n'.join(lines)
def failure(row):
    if not row['format_ok']: return '形式の崩れ'
    if row['gold_needs_review'] and row.get('both_ok') is False: return '根拠なしの質問への断定'
    return '根拠ありの質問への誤回答または不要な確認要求'
ex=EXERCISE_RESULT
ex_after=json.loads((EXERCISE_DIR/'eval_after.json').read_text())['rows']
lines=['# 第2回 解答','','## 演習','',
       '- 使ったモデル・設定：`'+json.dumps(ex['config'],ensure_ascii=False)+'`',
       '- 自分で学習を実行したか（executed_training_myself）：はい。このノートブックで学習処理を実行した。',
       '',table([ex['before'],ex['after']]),'',
       '- 退行した質問：'+('、'.join(ex['regressed']) or 'なし'),
       '- 学習後に残った失敗：']
for row in ex_after:
    if not row['both_ok']: lines.append(f"  - {row['question']}：{failure(row)}。出力：`{row['raw']}`")
if all(r['both_ok'] for r in ex_after): lines.append('  - 今回の評価21問ではなし。別の質問にも正答する保証ではない。')
lines+=['','### 知識を問う質問の結果','']
for row in ex['knowledge']:
    lines.append(f"- {row['質問']}：{row['学習前の分類']} → {row['学習後の分類']}。学習後の出力：`{row['学習後の出力']}`")
lines+=['',f"形式遵守率は{pct(ex['before']['format_rate'])}から{pct(ex['after']['format_rate'])}、内容正解率は{pct(ex['before']['content_rate'])}から{pct(ex['after']['content_rate'])}になった。形式と内容を別に測ることで、JSONに従えることと根拠に即して答えられることを区別できた。知識質問の分類と出力は上記のとおりであり、形式の学習だけから未知の質問への判断が保証されるとは結論しない。",'',
       '## 宿題','','- 取り組んだレベル：発展',
       '- 変えたデータ：講座の用語集58語から、指定スクリプトで116件のQ&Aを生成した。messages形式を維持し、回答をanswerとneeds_reviewのJSONに整え、用語集外の質問4件を追加した。学習120件、検証60件、評価18問とした。',
       '- 用語集外の用意方法：用語集の見出しと別名にない語を選び、学習・検証・評価で異なる語にした。「用語集にない」は一般知識として答えられないという意味ではない。用語集への収録の有無を判定対象とした。',
       '- 評価：用語名を含む質問6問、言い換え6問、用語集外6問。正解をgoldに分離し、モデルには渡さなかった。学習・検証・評価で質問文の重複はない。同じ用語を学習と評価で使うため、測っているのは既習用語の保持と言い換えへの対応であり、未知概念の学習ではない。',
       '- 設定：演習からnum_train_epochsを20→3、max_new_tokensを96→256に変更した。120件×3周、バッチ4×勾配累積2で45更新を予定した。LoRAのr=16、alpha=32、学習率2e-4、seed=42、量子化なし、モデルrevisionとPINSは講座の指定を維持した。回答が長い用語の打ち切りを減らすため、生成上限を増やし、すべての比較条件で同一にした。',
       f"- 実測：学習時間{training_seconds:.1f}秒、更新回数{trainer.state.global_step}、平均training loss {train_output.training_loss:.6f}、最後のログ区間loss {loss_logs[-1] if loss_logs else 'ログなし'}。最後のログ区間lossは厳密な最終バッチlossとは異なる。",'',
       '### 結果','',table([before_summary,after_summary,rag_summary,lookup_summary]),'',
       'before＝学習前・検索なし、after＝学習後・検索なし、retrieval_base＝学習前モデル＋検索結果、lookup_only＝検索した定義をそのまま返す方法。検索は用語名・別名の最長一致であり、意味検索ではない。', '',
       '### 質問の種類ごとの比較','']
for category in ['用語あり','言い換え','用語集外']:
    lines+=['#### '+category,'',table([{'label':s['label'],**s['categories'][category]} for s in [before_summary,after_summary,rag_summary,lookup_summary]]),'']
lines+=['### 出力の変化と残った失敗','']
for b,a in zip(before_rows,after_rows):
    g=gold[a['question']]
    lines.append(f"- {a['question']}：{b['classification']}（内容{'正解' if b['content_ok'] else '不正解'}）→ {a['classification']}（内容{'正解' if a['content_ok'] else '不正解'}）。学習後：`{a['raw']}`。正解例：`{g['answer']}`")
lines+=['','- 退行した質問：'+('、'.join(regressed) or 'なし'), '',
       '### 考察', '',
       f"今回、学習前後の内容正解率は{pct(before_summary['content_rate'])}→{pct(after_summary['content_rate'])}、形式遵守率は{pct(before_summary['format_rate'])}→{pct(after_summary['format_rate'])}だった。用語集外の内容正解率は{pct(before_summary['categories']['用語集外']['content_rate'])}→{pct(after_summary['categories']['用語集外']['content_rate'])}だった。全体の点数だけでなく、言い換えと用語集外を分けて確認した。",'',
       'ファインチューニングには、定型の答え方や専門用語の説明を小さいモデルに適応させ、毎回長い資料を渡さずに回答させられる可能性がある。一方、手本と評価データの準備、GPUによる学習、更新時の再学習が必要になる。学習データにない語への断定や、既習用語への誤回答が残る可能性もあるため、lossの低下だけで成功とは判断できない。', '',
       '用語集を検索する方法では、定義を更新すれば参照資料も更新でき、回答の根拠を確認しやすい。今回の単純な文字列検索は、用語名や別名を含まない言い換えを検索できない場合がある。そのため、この検索との得点差をもって意味検索を含むRAG全般の優劣とは結論しない。検索結果をモデルに渡す場合にも、検索成功と生成の正しさは別に評価する必要がある。', '',
       '頻繁に変わる講義情報や根拠の提示には検索が向いており、形式・口調・手順などの安定にはファインチューニングが候補になる。両者は併用できる。今回の18問は小規模で、既習概念の評価と必須語句による自動採点に限られる。言い換えとして正しい回答が不正解になることや、必須語句が含まれていても誤った説明が正解になることがあるので、出力原文を確認する。', '',
       '※自動生成した実測の要約。提出前に本人が出力原文と評価結果を確認し、自分の気づきを追記する。']
(WORK/'answer.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
print((WORK/'answer.md').read_text())


In [ ]:

import zipfile
archive=Path('/content/lesson02_submission.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder,prefix in [(WORK,'lectures/02/advanced'),(EXERCISE_DIR,'lectures/02/exercise')]:
        for p in folder.glob('*'):
            if p.is_file() and p.suffix in ['.json','.jsonl']: z.write(p,f'{prefix}/{p.name}')
    z.write(WORK/'answer.md','lectures/02/answer.md')
from google.colab import files
files.download(str(archive))
print('ダウンロード完了後、ランタイムを接続解除して削除してください。')
